### **Diffie-Hellman Key Exchange**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Diffie_Hellman_Key_Exchange.ipynb)

> **Diffie-Hellman (1976)** lets two parties agree on a shared secret over a public channel. Each one publishes `g^secret mod p`; both arrive at `g^(ab) mod p`, while an eavesdropper faces the **discrete logarithm** problem. Unauthenticated Diffie-Hellman, however, falls to a **man-in-the-middle**. The notebook ends with the modern form: **X25519** plus a key derivation function, feeding AES-GCM.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

In [1]:
# Classic Diffie-Hellman over a safe-prime group (64 bits for readability; real groups use 2048+ bits)
import secrets

def is_probable_prime(n, rounds=40):
    if n < 2:
        return False
    for small in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if n % small == 0:
            return n == small
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    for _ in range(rounds):
        x = pow(secrets.randbelow(n - 3) + 2, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def safe_prime(bits):
    while True:
        q = secrets.randbits(bits - 1) | (1 << (bits - 2)) | 1
        if is_probable_prime(q) and is_probable_prime(2 * q + 1):
            return 2 * q + 1

p = safe_prime(64)
g = 4                         # a square: generates the prime-order subgroup of size (p - 1) / 2

a = secrets.randbelow(p - 3) + 2          # Alice's secret
b = secrets.randbelow(p - 3) + 2          # Bob's secret
A, B = pow(g, a, p), pow(g, b, p)         # sent in the clear
alice_secret, bob_secret = pow(B, a, p), pow(A, b, p)
print(f"Public: p={p}, g={g}, A={A}, B={B}")
print(f"Alice computes {alice_secret}\nBob computes   {bob_secret}")
assert alice_secret == bob_secret

Public: p=13816957529571867887, g=4, A=9393087584210491618, B=10727979188818009603
Alice computes 1666634279717107862
Bob computes   1666634279717107862


#### **01. Man-in-the-Middle**

Diffie-Hellman proves nothing about **who** is on the other side. Mallory intercepts the exchange and runs one key agreement with Alice and another with Bob; she can then read and re-encrypt every message. Real protocols (TLS, SSH, Signal) **authenticate** the exchange with certificates or signatures.

In [2]:
# Mallory replaces A and B with her own public values
m = secrets.randbelow(p - 3) + 2
M = pow(g, m, p)
alice_key = pow(M, a, p)        # Alice believes she shares this with Bob
bob_key = pow(M, b, p)          # Bob believes he shares this with Alice
print("Alice and Bob share a key:", alice_key == bob_key)
print("Mallory knows Alice's key:", pow(A, m, p) == alice_key)
print("Mallory knows Bob's key  :", pow(B, m, p) == bob_key)

Alice and Bob share a key: False
Mallory knows Alice's key: True
Mallory knows Bob's key  : True


#### **02. Modern Practice: X25519 + HKDF + AES-GCM**

**X25519** is Diffie-Hellman on an elliptic curve (Curve25519): 32-byte keys with security comparable to 3072-bit classic groups. The raw shared secret is passed through **HKDF** to derive the actual encryption key.

In [3]:
# Key agreement with X25519, key derivation with HKDF, encryption with AES-GCM
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

alice_private, bob_private = X25519PrivateKey.generate(), X25519PrivateKey.generate()
alice_shared = alice_private.exchange(bob_private.public_key())
bob_shared = bob_private.exchange(alice_private.public_key())
assert alice_shared == bob_shared

def derive(shared):
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None, info=b"demo chat v1").derive(shared)

nonce = secrets.token_bytes(12)
sealed = AESGCM(derive(alice_shared)).encrypt(nonce, b"Hi Bob, this channel is encrypted.", None)
print("Bob reads:", AESGCM(derive(bob_shared)).decrypt(nonce, sealed, None).decode())

Bob reads: Hi Bob, this channel is encrypted.


#### **Key Takeaways**

* Diffie-Hellman agrees on a secret without sending it; its security rests on the discrete logarithm problem.
* Without authentication it falls to a man-in-the-middle.
* Prefer X25519 (or ECDH on P-256), always followed by a KDF, inside an authenticated protocol.